# DuoAlign tutorial — human lymph node slice2

| item | value |
|---|---|
| dataset | 10x Genomics Visium CytAssist, human lymph node section **slice2** |
| omics | transcriptome (RNA) + surface protein (ADT) |
| spots | 3359 |
| ground-truth domains | 10 |
| pretraining epochs | 350 |

**Pipeline** — one code cell per step, run top to bottom:

| cell | step |
|---|---|
| 0 | environment variables and project paths |
| 1 | imports |
| 2 | hyper-parameters |
| 3 | load data and build labels |
| 4 | preprocessing (HVG -> normalise -> log1p -> scale -> PCA) |
| 5 | graph construction (shared spatial graph + per-omics semantic graphs) |
| 6 | train DuoAlign (stage-1 pretraining + KMeans) |
| 7 | clustering metrics |
| 8 | spatial clustering map (ground truth vs DuoAlign) |
| 9 | cross-modal attention violin plot |
| 10 | UMAP of the learned joint embedding |

Figures are written to `DuoAlign/notebooks/figs/`.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Environment and paths
#    Fixed hash seed and thread counts, matching the official scripts in run_scripts/, for reproducibility.
# ---------------------------------------------------------------------------
import os
import sys

os.environ['PYTHONHASHSEED'] = '2026'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
for _v in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[_v] = '1'


def _find_project_root(start=None):
    """Walk upwards to the directory holding DuoAlign/model.py so the notebook runs from any cwd."""
    p = os.path.abspath(start or os.getcwd())
    for _ in range(8):
        if os.path.isfile(os.path.join(p, 'DuoAlign', 'model.py')):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    raise RuntimeError('Could not locate the project root automatically, please set DUOALIGN_ROOT')


# All three paths can be overridden through environment variables:
#   DUOALIGN_ROOT      project root (contains the DuoAlign/ package)
#   DUOALIGN_DATA_ROOT data root (defaults to <project root>/../data)
PROJECT_ROOT = os.environ.get('DUOALIGN_ROOT') or _find_project_root()
WORKSPACE_ROOT = os.path.dirname(PROJECT_ROOT)
DATA_ROOT = os.environ.get('DUOALIGN_DATA_ROOT') or os.path.join(WORKSPACE_ROOT, 'data')
FIG_DIR = os.path.join(PROJECT_ROOT, 'notebooks', 'figs')
os.makedirs(FIG_DIR, exist_ok=True)

# Make `import DuoAlign` work
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT    =', DATA_ROOT)
print('FIG_DIR      =', FIG_DIR)


In [ ]:
# ---------------------------------------------------------------------------
# 1. Imports
#    The three modules of DuoAlign itself: preprocess (graph construction / preprocessing) / train_duoalign (trainer) /
#    utils (evaluation metrics). All three come from the DuoAlign/ package added to sys.path above.
# ---------------------------------------------------------------------------
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib
try:
    # Switch to the inline backend inside Jupyter -> figures show up right below the cell
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    # Plain scripts / headless runs fall back to Agg; figures are still written to disk by savefig
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment   # Hungarian matching, used to align cluster colours
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder

from DuoAlign.preprocess import fix_seed, construct_duoalign_graphs
from DuoAlign.train_duoalign import Train_DuoAlign
from DuoAlign.utils import evaluate_clustering

warnings.filterwarnings('ignore', category=UserWarning)
sc.settings.verbosity = 1

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('scanpy', sc.__version__, '| torch', torch.__version__, '| device', DEVICE)


In [ ]:
# ---------------------------------------------------------------------------
# 2. Hyper-parameters
#    The defaults are the configuration used in the paper and in run_scripts/; usually no change needed.
#
#    graphs     spatial_k = 10      shared spatial graph: Euclidean-distance KNN
#               feature_k = 20      per-omics semantic graph: cosine-similarity KNN
#    latent     dim_out = 64
#    prototypes n_prototypes = 32, Sinkhorn eps = 0.05, iters = 50, tau_pcl = 0.1
#    loss       phi = 0.5  (phi weights the PSA and reconstruction terms, 1-phi weights PCL)
#               lambda_ortho = 0.01 (prototype orthogonality regulariser)
#    optimiser  lr = 1e-3, weight_decay = 5e-4
#    prep       n_top_genes = 3000, PCA dims: RNA 200 / second omics 50
# ---------------------------------------------------------------------------
class Args:
    dataset_dir = os.path.join(DATA_ROOT, 'Human_lymph_node', 'slice2')

    random_state = 2022        # shared random seed for KMeans and PCA
    use_dense_adj = True       # dense adjacency: strictly deterministic, recommended for reproduction

    dim_out = 64
    n_prototypes = 32
    tau_pcl = 0.1
    sinkhorn_eps = 0.05
    sinkhorn_iters = 50
    use_saf = True             # enable the self-adaptive fusion module (SAF)
    spatial_k = 10
    feature_k = 20

    lr = 1e-3
    weight_decay = 5e-4
    pretrain_epochs = 350
    phi = 0.5
    lambda_ortho = 0.01

    n_top_genes = 3000
    pca_comps_rna = 200
    pca_comps_pro = 50

    label_key = 'final_annot'

args = Args()
fix_seed(args.random_state)     # see DuoAlign/preprocess.py; seeds python/numpy/torch
print('spatial_k =', args.spatial_k, '| feature_k =', args.feature_k,
      '| epochs =', args.pretrain_epochs, '| seed =', args.random_state)


In [ ]:
# ---------------------------------------------------------------------------
# 3. Load data and labels
#    One h5ad per omics; the labels come straight from the RNA obs column final_annot.
#    Spot order of both omics is aligned to the RNA order so rows correspond one to one.
# ---------------------------------------------------------------------------
rna_path = os.path.join(args.dataset_dir, 's2_adata_rna.h5ad')
omics2_path = os.path.join(args.dataset_dir, 's2_adata_adt.h5ad')

adata_rna = sc.read_h5ad(rna_path)
adata_pro = sc.read_h5ad(omics2_path)
adata_rna.var_names_make_unique()
adata_pro.var_names_make_unique()

if args.label_key not in adata_rna.obs.columns:
    raise KeyError("label column '%s' not in obs; available: %s"
                   % (args.label_key, list(adata_rna.obs.columns)))

# 'manual-anno' is the ground-truth column name used consistently inside DuoAlign
adata_rna.obs['manual-anno'] = adata_rna.obs[args.label_key].astype(str)
adata_pro = adata_pro[adata_rna.obs_names].copy()

# Labels are encoded as 0..K-1; n_clusters is the KMeans cluster count (= number of true classes)
le = LabelEncoder()
adata_rna.obs['Group'] = le.fit_transform(adata_rna.obs['manual-anno'].astype(str))
n_clusters = len(le.classes_)
print('cells = %d | ground-truth classes = %d' % (adata_rna.n_obs, n_clusters))
print('labels:', list(le.classes_))


In [ ]:
# ---------------------------------------------------------------------------
# 4. Preprocessing
#    Spatial coordinates are min-max scaled to [0,1] before graph construction (graphs use relative positions only).
# ---------------------------------------------------------------------------
_coords = np.asarray(adata_rna.obsm['spatial'], dtype=np.float64)
adata_rna.obsm['spatial'] = _coords
coords_norm = ((_coords - _coords.min(axis=0)) /
               (_coords.max(axis=0) - _coords.min(axis=0)))

# --- omics 1: transcriptome -------------------------------------------------
# Highly variable genes -> library-size normalisation -> log1p -> clipped scaling -> PCA, stored in obsm['feat']
sc.pp.highly_variable_genes(adata_rna, flavor='seurat_v3', n_top_genes=args.n_top_genes)
sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)
sc.pp.scale(adata_rna, zero_center=False, max_value=10)
sc.pp.pca(adata_rna, n_comps=args.pca_comps_rna, use_highly_variable=True,
          svd_solver='arpack', random_state=args.random_state)
adata_rna.obsm['feat'] = adata_rna.obsm['X_pca'].copy()

# --- omics 2: protein (raw counts) -----------------------------------------
# Protein/ATAC follow the same path: normalisation -> log1p -> clipped scaling -> PCA.
# Dimension is min(50, n_vars-1): protein panels usually have fewer than 50 features, so the PCA dim shrinks automatically.
sc.pp.normalize_total(adata_pro, target_sum=1e4)
sc.pp.log1p(adata_pro)
sc.pp.scale(adata_pro, zero_center=False, max_value=10)
_n = min(args.pca_comps_pro, adata_pro.n_vars - 1)
if _n > 0:
    sc.pp.pca(adata_pro, n_comps=_n, svd_solver='arpack',
              random_state=args.random_state)
    adata_pro.obsm['feat'] = adata_pro.obsm['X_pca'].copy()
else:
    adata_pro.obsm['feat'] = np.asarray(adata_pro.X)

print('RNA feat:', adata_rna.obsm['feat'].shape,
      '| protein feat:', adata_pro.obsm['feat'].shape)


In [ ]:
# ---------------------------------------------------------------------------
# 5. Graph construction
#    construct_duoalign_graphs returns three graphs at once:
#      adj_spatial   spatial graph shared by both omics (Euclidean KNN, k=spatial_k), symmetrised + self-loops + D^-1/2 A D^-1/2
#      adj_feature1  semantic graph of omics 1 (cosine KNN, k=feature_k)
#      adj_feature2  semantic graph of omics 2 (same as above)
#    A shared spatial graph with independent semantic graphs is exactly what DuoAlign dual-view encoding assumes.
# ---------------------------------------------------------------------------
print('Building DuoAlign graphs (shared spatial + per-omics semantic)...')
graphs = construct_duoalign_graphs(coords_norm,
                                   adata_rna.obsm['feat'], adata_pro.obsm['feat'],
                                   spatial_k=args.spatial_k, feature_k=args.feature_k)
print('graph keys:', list(graphs.keys()))
for _k, _v in graphs.items():
    print('  %-14s shape = %s' % (_k, tuple(_v.shape)))


In [ ]:
# ---------------------------------------------------------------------------
# 6. Train DuoAlign
#    Train_DuoAlign takes {'adata_omics1','adata_omics2','graphs'} plus hyper-parameters and offers two entries:
#      pretrain()  -- stage one, single-omics pretraining (the only stage DuoAlign uses)
#    Once training ends, KMeans on z_cross yields the final spatial-domain labels.
# ---------------------------------------------------------------------------
data_dict = {'adata_omics1': adata_rna, 'adata_omics2': adata_pro, 'graphs': graphs}

trainer = Train_DuoAlign(data_dict, args)
print('=== Stage 1: single-omics pretraining (%d epochs) ===' % args.pretrain_epochs)
init_z = trainer.pretrain()          # returns z_cross from stage one

# KMeans protocol matches the paper: n_init=20, seed=2022, clusters = number of true classes
print('\n=== KMeans on the learned joint embedding ===')
kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=args.random_state)
final_labels = kmeans.fit_predict(init_z.cpu().numpy())

# ---------------------------------------------------------------------------
# Write results back into AnnData; keys keep the MHALGlue_ prefix so the workspace evaluation scripts work
#   obsm['MHALGlue_emb']          joint embedding z_cross  (used by the UMAP plot)
#   obs['MHALGlue_cluster']       cluster labels          (used by the cluster / violin plots)
#   obsm['MHALGlue_alpha_cross']  cross-modal attention weights (used by the violin plot)
#   obs['MHALGlue_eta1'/'eta2']   global fusion coefficients of the two views (diagnostics)
# ---------------------------------------------------------------------------
with torch.no_grad():
    trainer.model.eval()
    res_final = trainer._forward()

adata_rna.obsm['MHALGlue_emb'] = res_final['z_cross'].cpu().numpy()
adata_rna.obs['MHALGlue_cluster'] = pd.Categorical(final_labels)
if res_final.get('alpha_cross') is not None:
    adata_rna.obsm['MHALGlue_alpha_cross'] = res_final['alpha_cross'].cpu().numpy()
if res_final.get('eta1') is not None:
    adata_rna.obs['MHALGlue_eta1'] = float(np.asarray(res_final['eta1'].cpu()))
    adata_rna.obs['MHALGlue_eta2'] = float(np.asarray(res_final['eta2'].cpu()))

print('clusters:', len(np.unique(final_labels)), '| embedding:', adata_rna.obsm['MHALGlue_emb'].shape)


In [ ]:
# ---------------------------------------------------------------------------
# 7. Metrics
#    evaluate_clustering first Hungarian-matches predicted clusters to true classes, then computes the 10 metrics.
#    The paper text reports ARI / FMI / Completeness.
# ---------------------------------------------------------------------------
y_true = adata_rna.obs['Group'].values
metrics = evaluate_clustering(y_true, final_labels)

order = ['ARI', 'NMI', 'AMI', 'FMI', 'F1', 'Compl', 'Jaccard']
print('=' * 58)
print('DuoAlign on human lymph node slice2  --  clustering metrics')
print('=' * 58)
for _k in order:
    print('%-10s | %10.4f' % (_k, metrics[_k]))
for _k, _v in metrics.items():          # the remaining metrics (ACC/MI/V-Measure etc.)
    if _k not in order:
        print('%-10s | %10.4f' % (_k, _v))
print('=' * 58)

# Bar chart of the same metrics for a quick comparison
fig, ax = plt.subplots(figsize=(8, 4.2))
vals = [metrics[_k] for _k in order]
bars = ax.bar(order, vals, color=plt.cm.tab10(np.linspace(0, 1, len(order))),
              edgecolor='white', linewidth=0.8)
for _b, _v in zip(bars, vals):
    ax.text(_b.get_x() + _b.get_width() / 2, _v + 0.012, '%.3f' % _v,
            ha='center', fontsize=11)
ax.set_ylabel('Score', fontsize=13)
ax.set_ylim(0, min(1.05, max(vals) * 1.22 + 0.02))
ax.set_title('DuoAlign on human lymph node slice2', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
_p = os.path.join(FIG_DIR, '03_Human_lymph_node_slice2_metrics.png')
fig.savefig(_p, dpi=200, bbox_inches='tight')
plt.close(fig)
print('saved:', _p)

# Render the PNG written to disk: no matplotlib backend involved, so it always shows in the notebook
try:
    get_ipython()
    from IPython.display import Image, display
    display(Image(filename=_p))
except NameError:
    pass          # skipped when converted to .py and run with plain python


In [ ]:
# ---------------------------------------------------------------------------
# 8. Cluster visualisation (spatial maps)
#    Left: manual ground truth;  right: DuoAlign prediction.
#    Predicted clusters are Hungarian-matched to the true classes, so one colour means one spatial domain on both panels.
#    A predicted cluster matching no true class falls into the grey 'others' group.
# ---------------------------------------------------------------------------
def hungarian_map(y_true, y_pred):
    """Hungarian algorithm mapping predicted clusters to true classes: maximises the confusion-matrix diagonal sum."""
    D = max(int(y_pred.max()), int(y_true.max())) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    row, col = linear_sum_assignment(w.max() - w)
    mp = {r: c for r, c in zip(row, col)}
    return np.array([mp.get(p, D + p) for p in y_pred])


gt_names = np.unique(adata_rna.obs['manual-anno'].astype(str))
y_gt = np.searchsorted(gt_names, adata_rna.obs['manual-anno'].astype(str).values)
y_mapped = hungarian_map(y_gt, final_labels)

# Shared palette: the first K tab20 colours for the true classes, the last colour for 'others'
all_names = list(gt_names) + ['others']
palette = [plt.cm.tab20(i % 20) for i in range(len(all_names))]
palette[-1] = '#bbbbbb'

plot_gt = adata_rna.copy()
plot_gt.obs['_c'] = pd.Categorical(
    np.array([gt_names[m] if m < len(gt_names) else 'others' for m in y_gt], dtype=str),
    categories=all_names)
plot_da = adata_rna.copy()
plot_da.obs['_c'] = pd.Categorical(
    np.array([gt_names[m] if m < len(gt_names) else 'others' for m in y_mapped], dtype=str),
    categories=all_names)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
sc.pl.embedding(plot_gt, basis='spatial', color='_c', palette=palette,
                size=90, show=False, ax=axes[0], legend_loc=None, title='')
sc.pl.embedding(plot_da, basis='spatial', color='_c', palette=palette,
                size=90, show=False, ax=axes[1], legend_loc=None, title='')
for _ax, _t in zip(axes, ['Ground truth', 'DuoAlign (ARI: %.4f)' % metrics['ARI']]):
    _ax.set_aspect('auto')      # let the points fill the panel and avoid whitespace
    _ax.axis('off')
    _ax.set_title(_t, fontsize=17, fontweight='bold')
fig.tight_layout()
_p = os.path.join(FIG_DIR, '03_Human_lymph_node_slice2_spatial.png')
fig.savefig(_p, dpi=200, bbox_inches='tight')
plt.close(fig)
print('saved:', _p)

# Render the PNG written to disk: no matplotlib backend involved, so it always shows in the notebook
try:
    get_ipython()
    from IPython.display import Image, display
    display(Image(filename=_p))
except NameError:
    pass          # skipped when converted to .py and run with plain python


In [ ]:
# ---------------------------------------------------------------------------
# 9. Cross-modal attention violin plot
#    obsm['MHALGlue_alpha_cross'] has shape (N, 2):
#      A[:, 0] = weight of each spot on RNA
#      A[:, 1] = weight of each spot on protein
#    The two weights sum to 1 (softmax output), so a split violin is drawn per cluster label,
#    Shows at a glance which domains rely more on the transcriptome and which on protein.
# ---------------------------------------------------------------------------
if 'MHALGlue_alpha_cross' in adata_rna.obsm:
    import seaborn as sns

    A = np.asarray(adata_rna.obsm['MHALGlue_alpha_cross'])
    df = pd.DataFrame(columns=['RNA', 'protein', 'label'])
    df['RNA'], df['protein'] = A[:, 0], A[:, 1]
    df['label'] = adata_rna.obs['MHALGlue_cluster'].astype(str).values
    # wide to long format so it can drive seaborn hue
    df = df.set_index('label').stack().reset_index()
    df.columns = ['DuoAlign label', 'Modality', 'Attention weight']

    fig, ax = plt.subplots(figsize=(9, 4.8))
    sns.violinplot(data=df, x='DuoAlign label', y='Attention weight',
                   hue='Modality', split=True, inner='quart', linewidth=1,
                   cut=0, density_norm='width', bw_adjust=0.6, ax=ax)
    ax.set_title('RNA vs protein cross-modal attention', fontsize=15,
                 fontweight='bold')
    ax.set_xlabel('DuoAlign label', fontsize=13)
    ax.set_ylabel('Attention weight', fontsize=13)
    ax.legend(bbox_to_anchor=(1.22, 1.01), loc='upper right')
    fig.tight_layout()
    _p = os.path.join(FIG_DIR, '03_Human_lymph_node_slice2_attention_violin.png')
    fig.savefig(_p, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print('saved:', _p)

    # Render the PNG written to disk: no matplotlib backend involved, so it always shows in the notebook
    try:
        get_ipython()
        from IPython.display import Image, display
        display(Image(filename=_p))
    except NameError:
        pass          # skipped when converted to .py and run with plain python
else:
    print("obsm['MHALGlue_alpha_cross'] not found -- re-run the training cell above")


In [ ]:
# ---------------------------------------------------------------------------
# 10. UMAP
#     Left: ground-truth labels;  right: DuoAlign clusters.
#     Clusters that separate better and follow the left panel mean a more linearly separable embedding.
#     - Falls back to a 2-D PCA projection when umap-learn is missing, without raising.
#     - Note: size here controls the marker size, same meaning as in the spatial cluster plots above (unit pt^2).
# ---------------------------------------------------------------------------
emb = np.asarray(adata_rna.obsm['MHALGlue_emb'])
try:
    import umap
    red = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42,
                    n_components=2).fit_transform(emb)
    method = 'UMAP'
except Exception as _e:
    from sklearn.decomposition import PCA
    red = PCA(n_components=2, random_state=42).fit_transform(emb)
    method = 'PCA (umap-learn not installed)'
print('dim reduction:', method)

_ad = adata_rna.copy()
_ad.obsm['_red'] = red
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
sc.pl.embedding(_ad, basis='_red', color='manual-anno', palette=palette,
                size=20, show=False, ax=axes[0], legend_loc=None, title='')
sc.pl.embedding(_ad, basis='_red', color='MHALGlue_cluster', palette=palette,
                size=20, show=False, ax=axes[1], legend_loc='right margin',
                legend_fontsize=11, title='')
for _ax, _t in zip(axes, ['Ground truth', 'DuoAlign clusters']):
    _ax.set_aspect('auto')
    _ax.axis('off')
    _ax.set_title(_t, fontsize=18, fontweight='bold')
fig.tight_layout()
_p = os.path.join(FIG_DIR, '03_Human_lymph_node_slice2_umap.png')
fig.savefig(_p, dpi=200, bbox_inches='tight')
plt.close(fig)
print('saved:', _p)

# Render the PNG written to disk: no matplotlib backend involved, so it always shows in the notebook
try:
    get_ipython()
    from IPython.display import Image, display
    display(Image(filename=_p))
except NameError:
    pass          # skipped when converted to .py and run with plain python
